In [0]:
dbutils.widgets.text("src","")
source_value = dbutils.widgets.get("src")

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header","true")
    .option("cloudFiles.schemaLocation", f"/Volumes/workspace/bronze/bronzevolume/{source_value}/checkpoint")
    .option("cloudFiles.schemaEvolutionMode","rescue")
    .load(f"/Volumes/workspace/raw/rawvolume/rawdata/{source_value}/")
)


In [0]:
df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


In [0]:
(df.writeStream
.format("delta")
.outputMode("append")
.trigger(once=True)
.option("checkpointLocation", f"/Volumes/workspace/bronze/bronzevolume/{source_value}/checkpoint")
.option("path", f"/Volumes/workspace/bronze/bronzevolume/{source_value}/data")
.start()
)